# Gradient Boosting Exercises

This notebook contains hands-on exercises to help you understand Gradient Boosting.

## Learning Objectives
- Understand how Gradient Boosting fits residuals sequentially
- Explore the effect of hyperparameters (learning rate, n_estimators, max_depth)
- Compare with Random Forest and single Decision Trees
- Practice feature importance analysis
- Implement early stopping
- Apply to real-world-like problems

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Import our implementation
import sys
sys.path.append('.')
from implementation import GradientBoostingRegressor as OurGBR
from implementation import GradientBoostingClassifier as OurGBC

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## Exercise 1: Understanding Residual Fitting

**Goal**: Visualize how Gradient Boosting fits residuals step by step.

**Task**: 
1. Create a simple 1D regression dataset
2. Manually implement 3 boosting iterations
3. Visualize how predictions improve at each stage

In [ ]:
# Generate simple 1D data
np.random.seed(42)
X = np.linspace(0, 10, 100).reshape(-1, 1)
y = 2 * X.ravel() + 5 * np.sin(X.ravel()) + np.random.normal(0, 1, 100)

# TODO: Train a GradientBoostingRegressor with n_estimators=3
gb = GradientBoostingRegressor(n_estimators=3, learning_rate=1.0, max_depth=2, random_state=42)
gb.fit(X, y)

# TODO: Get staged predictions (predictions at each boosting stage)
predictions_by_stage = list(gb.staged_predict(X))

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot each stage
for i, ax in enumerate(axes.flat):
    ax.scatter(X, y, alpha=0.5, label='Actual data')
    if i < len(predictions_by_stage):
        ax.plot(X, predictions_by_stage[i], 'r-', linewidth=2, label=f'Prediction (Stage {i})')
        residuals = y - predictions_by_stage[i]
        rmse = np.sqrt(np.mean(residuals**2))
        ax.set_title(f'Stage {i}: RMSE = {rmse:.2f}')
    else:
        ax.set_title('Original Data')
    ax.legend()
    ax.set_xlabel('X')
    ax.set_ylabel('y')

plt.tight_layout()
plt.show()

# TODO: Calculate and print residuals at each stage
print("\nResidual Analysis:")
for i, pred in enumerate(predictions_by_stage):
    residuals = y - pred
    print(f"Stage {i}: Mean residual = {np.mean(residuals):.4f}, Std = {np.std(residuals):.4f}")

**Question**: How do the residuals change as we add more trees? Why?

## Exercise 2: Learning Rate Effect

**Goal**: Understand the trade-off between learning rate and number of trees.

**Task**: 
1. Generate synthetic sales data
2. Train models with different learning rates
3. Compare training and test performance

In [ ]:
# Generate store sales data
np.random.seed(42)
n_samples = 500
day_of_week = np.random.randint(0, 7, n_samples)
is_holiday = np.random.binomial(1, 0.1, n_samples)
temperature = np.random.normal(20, 8, n_samples)

X = np.column_stack([day_of_week, is_holiday, temperature])
sales = (
    1000 + 
    150 * (day_of_week >= 5) + 
    300 * is_holiday + 
    10 * temperature - 0.2 * temperature**2 +
    np.random.normal(0, 50, n_samples)
)

X_train, X_test, y_train, y_test = train_test_split(X, sales, test_size=0.2, random_state=42)

# TODO: Train models with different learning rates
learning_rates = [0.01, 0.05, 0.1, 0.5, 1.0]
n_estimators = 200

results = []
for lr in learning_rates:
    gb = GradientBoostingRegressor(
        n_estimators=n_estimators,
        learning_rate=lr,
        max_depth=3,
        random_state=42
    )
    gb.fit(X_train, y_train)
    
    # Get staged predictions
    train_scores = []
    test_scores = []
    
    for train_pred, test_pred in zip(gb.staged_predict(X_train), gb.staged_predict(X_test)):
        train_scores.append(mean_squared_error(y_train, train_pred))
        test_scores.append(mean_squared_error(y_test, test_pred))
    
    results.append({
        'lr': lr,
        'train_scores': train_scores,
        'test_scores': test_scores
    })

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

for result in results:
    ax1.plot(result['train_scores'], label=f"LR={result['lr']}")
    ax2.plot(result['test_scores'], label=f"LR={result['lr']}")

ax1.set_xlabel('Number of Trees')
ax1.set_ylabel('Train MSE')
ax1.set_title('Training Error vs Number of Trees')
ax1.legend()
ax1.set_yscale('log')

ax2.set_xlabel('Number of Trees')
ax2.set_ylabel('Test MSE')
ax2.set_title('Test Error vs Number of Trees')
ax2.legend()
ax2.set_yscale('log')

plt.tight_layout()
plt.show()

# TODO: Print final performance
print("\nFinal Test RMSE for different learning rates:")
for result in results:
    final_rmse = np.sqrt(result['test_scores'][-1])
    print(f"Learning Rate {result['lr']:4.2f}: ${final_rmse:.2f}")

**Questions**:
1. Which learning rate achieves the best test performance?
2. What happens with very high learning rates?
3. Why do lower learning rates need more trees?

## Exercise 3: Comparison with Random Forest and Decision Trees

**Goal**: Compare Gradient Boosting with other tree-based methods.

**Task**: 
1. Train Decision Tree, Random Forest, and Gradient Boosting
2. Compare performance on customer churn prediction
3. Analyze strengths and weaknesses

In [ ]:
# Generate customer churn data
np.random.seed(42)
n_samples = 1000

tenure = np.random.exponential(24, n_samples)
monthly_charges = np.random.uniform(30, 150, n_samples)
num_complaints = np.random.poisson(2, n_samples)

X = np.column_stack([tenure, monthly_charges, num_complaints])

logit = -2 - 0.05 * tenure + 0.02 * monthly_charges + 0.5 * num_complaints
churn_prob = 1 / (1 + np.exp(-logit))
y = (np.random.random(n_samples) < churn_prob).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TODO: Train models
models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, 
                                                     max_depth=3, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    results[name] = accuracy
    
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print(f"Accuracy: {accuracy:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Not Churn', 'Churn']))

# Visualize comparison
plt.figure(figsize=(10, 6))
plt.bar(results.keys(), results.values())
plt.ylabel('Accuracy')
plt.title('Model Comparison: Customer Churn Prediction')
plt.ylim([0.5, 1.0])
for i, (name, acc) in enumerate(results.items()):
    plt.text(i, acc + 0.01, f'{acc:.3f}', ha='center')
plt.tight_layout()
plt.show()

**Questions**:
1. Which model performs best and why?
2. Why does Gradient Boosting typically outperform Random Forest?
3. When might you prefer Random Forest over Gradient Boosting?

## Exercise 4: Feature Importance Analysis

**Goal**: Analyze and interpret feature importance.

**Task**: 
1. Train model on multi-feature dataset
2. Extract and visualize feature importance
3. Compare importance from different models

In [ ]:
# Generate retail dataset with meaningful features
np.random.seed(42)
n_samples = 1000

# Create features with different levels of importance
store_size = np.random.uniform(500, 5000, n_samples)  # High importance
location_score = np.random.uniform(0, 10, n_samples)  # High importance
parking_spaces = np.random.randint(10, 200, n_samples)  # Medium importance
years_open = np.random.uniform(0, 20, n_samples)  # Medium importance
competitor_distance = np.random.exponential(2, n_samples)  # Low importance
random_noise = np.random.normal(0, 1, n_samples)  # No importance

X = np.column_stack([store_size, location_score, parking_spaces, 
                     years_open, competitor_distance, random_noise])
feature_names = ['Store Size', 'Location Score', 'Parking Spaces', 
                'Years Open', 'Competitor Distance', 'Random Noise']

# Target: Monthly revenue
revenue = (
    5000 +
    2 * store_size +  # Main driver
    800 * location_score +  # Main driver
    20 * parking_spaces +  # Medium effect
    100 * years_open -  # Medium effect
    50 * competitor_distance +  # Small effect
    0 * random_noise +  # No effect
    np.random.normal(0, 1000, n_samples)
)

X_train, X_test, y_train, y_test = train_test_split(X, revenue, test_size=0.2, random_state=42)

# TODO: Train models and extract feature importance
models = {
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, 
                                                    max_depth=3, random_state=42)
}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for ax, (name, model) in zip(axes, models.items()):
    model.fit(X_train, y_train)
    importances = model.feature_importances_
    
    # Sort by importance
    indices = np.argsort(importances)[::-1]
    
    ax.bar(range(len(importances)), importances[indices])
    ax.set_xticks(range(len(importances)))
    ax.set_xticklabels([feature_names[i] for i in indices], rotation=45, ha='right')
    ax.set_ylabel('Importance')
    ax.set_title(f'{name}\nFeature Importance')
    
    # Print importance
    print(f"\n{name} Feature Importance:")
    for i in indices:
        print(f"  {feature_names[i]:20s}: {importances[i]:.4f}")

plt.tight_layout()
plt.show()

**Questions**:
1. Which features are most important? Does this match the data generation process?
2. How do feature importances differ between Random Forest and Gradient Boosting?
3. Why is "Random Noise" near zero importance?

## Exercise 5: Early Stopping

**Goal**: Implement and understand early stopping.

**Task**: 
1. Train model with early stopping
2. Compare with model without early stopping
3. Visualize validation curves

In [ ]:
# Use our from-scratch implementation which supports early stopping
X_train_full, X_test, y_train_full, y_test = train_test_split(X, revenue, test_size=0.2, random_state=42)

# TODO: Train with early stopping
gb_early = OurGBR(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=3,
    early_stopping_rounds=10,
    validation_fraction=0.2,
    random_state=42,
    verbose=1
)
gb_early.fit(X_train_full, y_train_full)

# Train without early stopping
gb_full = OurGBR(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=3,
    early_stopping_rounds=None,
    random_state=42,
    verbose=0
)
gb_full.fit(X_train_full, y_train_full)

# Visualize
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(gb_early.train_losses_, label='Training Loss', linewidth=2)
if gb_early.val_losses_:
    plt.plot(gb_early.val_losses_, label='Validation Loss', linewidth=2)
    plt.axvline(gb_early.n_estimators_, color='r', linestyle='--', label='Early Stop Point')
plt.xlabel('Boosting Iteration')
plt.ylabel('Mean Squared Error')
plt.title('Model with Early Stopping')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(gb_full.train_losses_, label='Training Loss', linewidth=2)
plt.xlabel('Boosting Iteration')
plt.ylabel('Mean Squared Error')
plt.title('Model without Early Stopping')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Compare test performance
y_pred_early = gb_early.predict(X_test)
y_pred_full = gb_full.predict(X_test)

rmse_early = np.sqrt(mean_squared_error(y_test, y_pred_early))
rmse_full = np.sqrt(mean_squared_error(y_test, y_pred_full))

print(f"\nTest RMSE with early stopping ({gb_early.n_estimators_} trees): ${rmse_early:.2f}")
print(f"Test RMSE without early stopping (300 trees): ${rmse_full:.2f}")
print(f"\nEarly stopping saved {300 - gb_early.n_estimators_} trees!")

**Questions**:
1. How many trees were actually used with early stopping?
2. Is test performance better with early stopping?
3. What are the benefits of early stopping?

## Exercise 6: Hyperparameter Tuning Challenge

**Goal**: Find optimal hyperparameters for a dataset.

**Task**: 
1. Use GridSearchCV to tune hyperparameters
2. Evaluate on a hold-out test set
3. Compare with default parameters

In [ ]:
from sklearn.model_selection import GridSearchCV

# Use the churn dataset from Exercise 3
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# TODO: Define parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [2, 3, 4, 5],
    'subsample': [0.8, 1.0]
}

# TODO: Perform grid search
print("Performing grid search (this may take a minute...)\n")
gb = GradientBoostingClassifier(random_state=42)
grid_search = GridSearchCV(
    gb, 
    param_grid, 
    cv=3, 
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

print("\nBest Parameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest Cross-Validation Score: {grid_search.best_score_:.4f}")

# Test on hold-out set
best_model = grid_search.best_estimator_
test_score = best_model.score(X_test, y_test)
print(f"Test Set Accuracy (Tuned): {test_score:.4f}")

# Compare with default
default_gb = GradientBoostingClassifier(random_state=42)
default_gb.fit(X_train, y_train)
default_score = default_gb.score(X_test, y_test)
print(f"Test Set Accuracy (Default): {default_score:.4f}")

print(f"\nImprovement: {(test_score - default_score) * 100:.2f} percentage points")

**Questions**:
1. What are the optimal hyperparameters found?
2. How much improvement over default parameters?
3. Which hyperparameter had the biggest impact?

## Bonus Exercise: Your Own Dataset

**Challenge**: Apply Gradient Boosting to your own dataset!

**Steps**:
1. Load your dataset
2. Prepare features and target
3. Train Gradient Boosting model
4. Tune hyperparameters
5. Analyze results and feature importance

In [ ]:
# Your code here!
# TODO: Load your dataset
# TODO: Preprocess and split
# TODO: Train and evaluate Gradient Boosting
# TODO: Compare with other models
# TODO: Analyze feature importance

print("Ready for your own experiments!")

## Summary

In these exercises, you learned:

1. **Residual Fitting**: How Gradient Boosting iteratively fits residuals
2. **Learning Rate**: The trade-off between learning rate and number of trees
3. **Model Comparison**: Strengths of GB vs Random Forest vs Decision Trees
4. **Feature Importance**: How to extract and interpret feature importance
5. **Early Stopping**: Preventing overfitting with validation-based stopping
6. **Hyperparameter Tuning**: Finding optimal parameters with grid search

### Key Takeaways:

- Gradient Boosting builds models sequentially, each correcting previous errors
- Lower learning rates with more trees generally perform better
- Early stopping helps prevent overfitting
- Feature importance reveals which features drive predictions
- Hyperparameter tuning is crucial for optimal performance

### Next Steps:

- Explore XGBoost and LightGBM for faster, more powerful gradient boosting
- Try different loss functions for specific problems
- Apply to your own datasets and problems
- Experiment with more advanced techniques like stacking and blending